# Hypothesis Testing
**Week 5 · Wednesday · 10:00–13:00**  
**Learning path:** CLT → hypotheses → manual statistic → critical values and p-value → Matplotlib → SciPy → interpretation.

**Estimated teaching time:** ~150 minutes plus 30 minutes of buffer. The final recap can be assigned as independent practice if needed.

By the end, you should be able to explain what the p-value measures, calculate it using yesterday's `cdf` and `sf`, make the same decision using critical values, and verify the result with SciPy.

> We do not use a p-value to calculate the probability that the null hypothesis is true. Rejecting $H_0$ does not establish causality.

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt

alpha = 0.05
import ipywidgets as widgets
from IPython.display import display, clear_output


## 1. Central Limit Theorem (CLT) — 15 minutes

A *sampling distribution* describes how a statistic changes across repeated random samples. For independent observations with finite variance, as the sample size grows, the distribution of the **sample mean** approaches a normal distribution, even if the population is not normal.

$$E(\bar X)=\mu,\qquad SE(\bar X)=\frac{\sigma}{\sqrt n}.$$

**Notation:**

- $X_i$: the $i$-th observation in a random sample.
- $\bar X=\frac{1}{n}\sum_{i=1}^{n}X_i$: the **sample mean as a random variable** (before observing the sample).
- $E(\bar X)$: the **expected value** of the sample mean — intuitively, the **mean of all the sample means** obtained from repeated random samples of the same size. More precisely, it is the probability-weighted mean over the sampling distribution (not necessarily a finite list of samples).
- $\mu$: the **population mean**.
- $\sigma$: the **population standard deviation**.
- $n$: the **sample size**.
- $SE(\bar X)$: the **standard error**, i.e. the standard deviation of the sampling distribution of $\bar X$.
- $\bar x$: the **observed sample mean** computed from a particular dataset.

When $\sigma$ is unknown, we estimate the standard error using $s/\sqrt n$, where $s$ is the **sample standard deviation** (computed with `ddof=1`), not the population standard deviation $\sigma$.

The CLT is not a guarantee for every sample size or every population. Independence and extreme tails matter.

In [ ]:
rng = np.random.default_rng(42)
population = st.expon(scale=10).rvs(size=100_000, random_state=rng)

fig, ax = plt.subplots(figsize=(10, 5))
for sample_size in [2, 10, 50]:
    means = np.array([
        rng.choice(population, size=sample_size).mean()
        for _ in range(1000)
    ])
    ax.hist(means, bins=40, density=True, alpha=0.42,
            label=f"n = {sample_size}")

ax.set(title="Sampling distributions of the mean",
       xlabel="Sample mean", ylabel="Density")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.show()

**Check for understanding (3 min).** If sample size increases from 25 to 100, by what factor does the standard error change? Does the CLT claim the *original population* becomes normal?

## 2. Anatomy of a hypothesis test — 15 minutes

1. Define the population parameter. State $H_1$ to express **the claim for which we seek evidence**, then write $H_0$ as the reference hypothesis. We do not establish $H_1$ merely by failing to reject $H_0$.
2. Choose $\alpha$ **before** examining the result.
3. Select an appropriate statistic and its distribution **under $H_0$**.
4. Compute the observed statistic manually.
5. Decide using **critical values** or a **p-value**.
6. Confirm with SciPy and interpret the result in context.

For a statistic $T$ observed as $t_{obs}$:

- **Greater:** $\text{p-value}=P(T\ge t_{obs}\mid H_0)=\mathrm{SF}(t_{obs})=1-\mathrm{CDF}(t_{obs})$.
- **Less:** $\text{p-value}=P(T\le t_{obs}\mid H_0)=\mathrm{CDF}(t_{obs})$.
- **Two-sided, symmetric null distribution:** $\text{p-value}=P(|T|\ge|t_{obs}|\mid H_0)=2\mathrm{SF}(|t_{obs}|)=2[1-\mathrm{CDF}(|t_{obs}|)]$.

For continuous distributions, $<$ and $\le$ have equal probability. **The two-sided shortcut requires symmetry**; do not automatically apply it to an $F$ or $\chi^2$ distribution.

**Visualization note:** Several examples below use explicitly labelled synthetic observations. They are chosen to keep p-value areas visible while preserving the intended decisions. Real observations must never be altered to obtain a desired significance result.

### Visual tool: critical regions and p-value areas

- **Red shaded areas:** rejection regions, determined by the significance level $\alpha$.
- **Red vertical lines:** critical values. They stop **at the null distribution curve**.
- **Blue shaded areas:** tail probabilities contributing to the **p-value**. They remain fixed when $\alpha$ changes.
- **Black vertical line:** the **observed test statistic**. It also stops at the curve and remains fixed.
- **Horizontal arrows above the curve:** show which area is accumulated, with its **CDF** or **SF** expression. For two-sided tests, both tails contribute.

For a right-tailed test, $\text{p-value}=\mathrm{SF}(t_{obs})=1-\mathrm{CDF}(t_{obs})$; for a left-tailed test, $\text{p-value}=\mathrm{CDF}(t_{obs})$. For a symmetric two-sided test, $\text{p-value}=2\mathrm{SF}(|t_{obs}|)$.

**Interactive significance level.** Move the slider from 0.001 to 0.100 (default 0.050). The **red boundaries and the decision** change, but the **observed statistic and its p-value do not**.

**Important:** Choose $\alpha$ **before** examining the results. The slider is an educational tool for exploring decision rules, not a way to select a convenient significance level after seeing the p-value.

In [ ]:
def _draw_statistical_test(statistic, alpha=0.05, distribution="t", df=None,
                           alternative="two-sided", title=None):
    """Draw an interactive hypothesis-test diagram using Matplotlib.

    Red = rejection region / critical boundaries; blue = p-value area;
    black = observed statistic. All vertical markers stop at the density.
    """
    if distribution in {"normal", "norm"}:
        dist = st.norm()
    elif distribution == "t":
        dist = st.t(df=df)
    elif distribution == "f":
        dist = st.f(dfn=df[0], dfd=df[1])
    elif distribution == "chi2":
        dist = st.chi2(df=df)
    else:
        raise ValueError("Unknown distribution")
    if alternative not in {"two-sided", "greater", "less"}:
        raise ValueError("Unknown alternative")
    if alternative == "two-sided" and distribution not in {"t", "normal", "norm"}:
        raise ValueError("Two-sided shortcut requires a symmetric null distribution")

    # Keep the horizontal axis fixed while alpha changes, so the change in
    # the rejection regions is easy to observe without rescaling the plot.
    lo, hi = float(dist.ppf(0.00025)), float(dist.isf(0.00025))
    if alternative == "two-sided":
        lo = min(lo, -abs(statistic) - 0.5)
        hi = max(hi, abs(statistic) + 0.5)
    else:
        lo = min(lo, statistic - 0.5)
        hi = max(hi, statistic + 0.5)
    x = np.linspace(lo, hi, 3000)
    y = dist.pdf(x)

    symbol = "z" if distribution in {"normal", "norm"} else ("F" if distribution == "f" else ("chi2" if distribution == "chi2" else "t"))
    if alternative == "two-sided":
        critical = [float(dist.ppf(alpha / 2)), float(dist.isf(alpha / 2))]
        observed_tails = [-abs(statistic), abs(statistic)]
        critical_mask = (x <= critical[0]) | (x >= critical[1])
        p_mask = (x <= observed_tails[0]) | (x >= observed_tails[1])
        p_value = 2 * float(dist.sf(abs(statistic)))
        tails = [("left", observed_tails[0], f"CDF(-|{symbol}_obs|)"),
                 ("right", observed_tails[1], f"SF(|{symbol}_obs|)")]
    elif alternative == "greater":
        critical = [float(dist.isf(alpha))]
        observed_tails = [statistic]
        critical_mask = x >= critical[0]
        p_mask = x >= statistic
        p_value = float(dist.sf(statistic))
        tails = [("right", statistic, f"SF({symbol}_obs) = 1 - CDF({symbol}_obs)")]
    else:
        critical = [float(dist.ppf(alpha))]
        observed_tails = [statistic]
        critical_mask = x <= critical[0]
        p_mask = x <= statistic
        p_value = float(dist.cdf(statistic))
        tails = [("left", statistic, f"CDF({symbol}_obs)")]

    fig, ax = plt.subplots(figsize=(10.5, 5.3))
    # Shade first, then draw the null density and all markers above the fills.
    ax.fill_between(x, 0, y, where=critical_mask, color="firebrick", alpha=0.20,
                    interpolate=True, label="Rejection region (alpha)")
    ax.fill_between(x, 0, y, where=p_mask, color="steelblue", alpha=0.48,
                    interpolate=True, label="p-value area")
    ax.plot(x, y, color="#253347", lw=2.1, label="Null distribution", zorder=4)

    # Vertical markers end exactly on the density curve, never above it.
    for i, cut in enumerate(critical):
        ax.vlines(cut, 0, float(dist.pdf(cut)), color="firebrick", lw=2,
                  zorder=6, label="Critical value" if i == 0 else None)
    ax.vlines(statistic, 0, float(dist.pdf(statistic)), color="black", lw=2.3,
              zorder=7, label="Observed statistic")

    # Label the x coordinates below the axis: red critical values and black
    # observed statistic, including when they lie close to each other.
    axis_transform = ax.get_xaxis_transform()
    for cut in critical:
        ax.text(cut, -0.085, f"{cut:.3f}", color="firebrick", ha="center",
                va="top", fontsize=9, transform=axis_transform, clip_on=False)
    ax.text(statistic, -0.17, f"{statistic:.3f}", color="black", ha="center",
            va="top", fontsize=9, fontweight="bold", transform=axis_transform,
            clip_on=False)

    # Probability-area arrows occupy the space ABOVE the density. They do not
    # intersect the red critical markers, which stop at the curve.
    ymax = float(np.max(y))
    ax.set_ylim(0, ymax * 1.62)
    arrow_y = ymax * 1.31
    label_y = ymax * 1.39
    for side, boundary, formula in tails:
        end = lo + 0.018 * (hi - lo) if side == "left" else hi - 0.018 * (hi - lo)
        ax.annotate("", xy=(end, arrow_y), xytext=(boundary, arrow_y),
                    arrowprops=dict(arrowstyle="->", color="steelblue", lw=1.8))
        ax.text((boundary + end) / 2, label_y, formula, ha="center",
                color="steelblue", fontsize=10)
    if alternative == "two-sided":
        ax.text((lo + hi) / 2, ymax * 1.54,
                f"p-value = CDF(-|{symbol}_obs|) + SF(|{symbol}_obs|) = {p_value:.3f}",
                ha="center", color="steelblue", fontsize=10, fontweight="bold")
    else:
        ax.text((lo + hi) / 2, ymax * 1.54, f"p-value = {p_value:.3f}",
                ha="center", color="steelblue", fontsize=10, fontweight="bold")

    ax.set_xlim(lo, hi)
    ax.set(xlabel="Test statistic under H0", ylabel="Probability density",
           title=title or f"{distribution} test")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.50), fontsize=8, frameon=False)
    fig.subplots_adjust(bottom=0.25, top=0.91, left=0.09, right=0.77)
    plt.show()

    decision = "Reject H0" if p_value < alpha else "Do not reject H0"
    print(f"alpha = {alpha:.3f} | observed statistic = {statistic:.3f} | "
          f"p-value = {p_value:.3f} | Decision: {decision}")
    return p_value, critical


def show_statistical_test(statistic, alpha=0.05, distribution="t", df=None,
                          alternative="two-sided", title=None):
    """Show a Matplotlib figure controlled by an ipywidgets alpha slider."""
    alpha_slider = widgets.FloatSlider(
        value=float(alpha), min=0.001, max=0.100, step=0.001,
        description="alpha", readout_format=".3f", continuous_update=False,
        style={"description_width": "45px"},
        layout=widgets.Layout(width="520px")
    )

    def refresh(alpha):
        _draw_statistical_test(statistic, alpha=alpha, distribution=distribution,
                               df=df, alternative=alternative, title=title)

    output = widgets.interactive_output(refresh, {"alpha": alpha_slider})
    display(widgets.VBox([output, alpha_slider]))
    return None


## 3. One-sample t-test — 30 minutes

**Illustrative synthetic scenario:** A travel company asks whether the mean fare for a premium ticket differs from $65. We use a simulated sample specifically designed to make both p-value tails visible. These are not real Titanic fares.

$H_0:\mu=65$; $H_1:\mu\ne65$. This is a **two-sided** test because observations can contradict $H_0$ in **two directions**: the population mean could be **greater than 65** or **less than 65**. We are looking for evidence of a difference, not specifically an increase or decrease.

**Put the claim we want to investigate in $H_1$.** The null $H_0$ provides the reference value used to calculate the test statistic and its distribution.

Here $\bar x$ is the observed sample mean; $s$ is the **sample standard deviation** (`ddof=1`), whereas $\sigma$ denotes the **population standard deviation**. We use $s$ because $\sigma$ is unknown.

$$t_{obs}=\frac{\bar x-\mu_0}{s/\sqrt n},\qquad df=n-1.$$

**Critical-value strategy:** reject when $t_{obs}<t_{\alpha/2,df}$ or $t_{obs}>t_{1-\alpha/2,df}$.

**p-value strategy (before the figure):**

$$\text{p-value}=P(|T|\ge|t_{obs}|\mid H_0)=2P(T\ge|t_{obs}|)=2[1-\mathrm{CDF}(|t_{obs}|)]=2\mathrm{SF}(|t_{obs}|).$$

In [ ]:
# Synthetic premium-ticket fares; fixed seed for reproducibility.
rng_fares = np.random.default_rng(2026)
first_class = rng_fares.normal(72, 35, size=100)
# Shift the sample mean to obtain a moderately significant statistic (not a tiny p-value).
first_class = first_class - first_class.mean() + 65 + 2.35 * first_class.std(ddof=1) / np.sqrt(len(first_class))
mu0 = 65
n1 = len(first_class)
xbar = first_class.mean()
s = first_class.std(ddof=1)
t_obs = (xbar - mu0) / (s / np.sqrt(n1))
df1 = n1 - 1
lower = st.t.ppf(alpha / 2, df=df1)
upper = st.t.isf(alpha / 2, df=df1)
p_value_manual = 2 * st.t.sf(abs(t_obs), df=df1)
print(f"n = {n1}, sample mean = {xbar:.3f}, s = {s:.3f}")
print(f"t = {t_obs:.3f}, critical interval = ({lower:.3f}, {upper:.3f})")
print(f"p-value = 2 * SF(|t|) = {p_value_manual:.3f}")
print(f"Critical decision: {'Reject H0' if t_obs < lower or t_obs > upper else 'Do not reject H0'}")
print(f"p-value decision: {'Reject H0' if p_value_manual < alpha else 'Do not reject H0'}")

In [ ]:
show_statistical_test(t_obs, alpha, distribution="t", df=df1,
                      alternative="two-sided", title="One-sample t-test: two tails")

In [ ]:
result = st.ttest_1samp(first_class, popmean=mu0, alternative="two-sided")
print(f"SciPy: t = {result.statistic:.3f}, p-value = {result.pvalue:.3f}")
assert np.isclose(result.statistic, t_obs)
assert np.isclose(result.pvalue, p_value_manual)

### One sample, one-sided alternative

Now ask whether the mean premium-ticket fare is **greater** than $65$: $H_1:\mu>65$.

$$\text{p-value}=P(T\ge t_{obs}\mid H_0)=1-\mathrm{CDF}(t_{obs})=\mathrm{SF}(t_{obs}).$$

Critical strategy: reject when $t_{obs}>\mathrm{PPF}(1-\alpha)=\mathrm{ISF}(\alpha)$.

In [ ]:
p_value_greater = st.t.sf(t_obs, df=df1)
critical_greater = st.t.isf(alpha, df=df1)
print(f"Critical = {critical_greater:.3f}, p-value = {p_value_greater:.3f}")
show_statistical_test(t_obs, alpha, distribution="t", df=df1, alternative="greater")
print(f"SciPy p-value = {st.ttest_1samp(first_class, mu0, alternative='greater').pvalue:.3f}")

**Check for understanding (4 min).** If the observed statistic is negative, would changing from a two-sided test to a `greater` test make the p-value smaller or larger? Why?

## 4. Independent two-sample t-test (Welch) — 25 minutes

**Illustrative synthetic scenario:** Compare the mean spending of two independent customer groups. Their population variances need not be equal. The simulated samples have different spreads and a moderately significant mean difference.

Begin with the hypotheses expressed directly in terms of the two population means:

$$H_0:\mu_m=\mu_f,\qquad H_1:\mu_m\ne\mu_f.$$

Equivalently, define $\delta$ as the hypothesized difference under $H_0$:

$$H_0:\mu_m-\mu_f=\delta,\qquad H_1:\mu_m-\mu_f\ne\delta.$$

For the question **“Are the means equal?”**, we set $\delta=0$. A nonzero $\delta$ would test a different claim, such as whether one population mean exceeds the other by a specified amount.

Use **Welch's test**: independent groups, no equal-variance assumption.

$$t_{obs}=\frac{(\bar x_m-\bar x_f)-\delta}{\sqrt{s_m^2/n_m+s_f^2/n_f}}.$$

Welch–Satterthwaite degrees of freedom:

$$\nu=\frac{(s_m^2/n_m+s_f^2/n_f)^2}{(s_m^2/n_m)^2/(n_m-1)+(s_f^2/n_f)^2/(n_f-1)}.$$

**Degrees of freedom note:** $\nu$ is an *approximation* and is generally **not an integer**. SciPy accepts and uses the fractional value directly. If a traditional printed t-table requires integer degrees of freedom, an integer approximation may be used (often rounding **down** conservatively); do **not** round up as a requirement of Welch's test.

**p-value before figure:** $\text{p-value}=P(|T_\nu|\ge|t_{obs}|)=2[1-\mathrm{CDF}_\nu(|t_{obs}|)]=2\mathrm{SF}_\nu(|t_{obs}|)$.

**Critical strategy:** compare $t_{obs}$ with $\mathrm{PPF}_\nu(\alpha/2)$ and $\mathrm{ISF}_\nu(\alpha/2)$.

In [ ]:
# Synthetic independent groups with different variances (Welch test).
rng_welch = np.random.default_rng(73)
male = rng_welch.normal(60, 25, size=80)
female = rng_welch.normal(60, 42, size=85)
nm, nf = len(male), len(female)
vm, vf = male.var(ddof=1), female.var(ddof=1)
a, b = vm / nm, vf / nf
delta = 0  # Null-hypothesis difference: equal population means
# Shift one sample to yield a visible, moderately significant two-tailed p-value.
male = male - male.mean() + female.mean() + 2.4*np.sqrt(a+b)
t_welch = ((male.mean() - female.mean()) - delta) / np.sqrt(a + b)
df_welch = (a + b)**2 / (a*a/(nm-1) + b*b/(nf-1))
p_value_welch = 2 * st.t.sf(abs(t_welch), df=df_welch)
lo_w = st.t.ppf(alpha/2, df=df_welch)
hi_w = st.t.isf(alpha/2, df=df_welch)
print(f"t = {t_welch:.3f}, Welch df = {df_welch:.3f}, p-value = {p_value_welch:.3f}")
print(f"Critical interval = ({lo_w:.3f}, {hi_w:.3f})")

In [ ]:
show_statistical_test(t_welch, alpha, distribution="t", df=df_welch,
                      alternative="two-sided", title="Welch independent t-test")
welch_result = st.ttest_ind(male, female, equal_var=False)
print(f"SciPy: t = {welch_result.statistic:.3f}, p-value = {welch_result.pvalue:.3f}")
assert np.isclose(welch_result.statistic, t_welch)
assert np.isclose(welch_result.pvalue, p_value_welch)

**Teaching note.** Welch does not assume equal population variances. Its effective degrees of freedom are approximated by the Welch–Satterthwaite formula and may be fractional. The derivation is beyond the scope of this lesson.

## 5. Paired t-test — 20 minutes

**Illustrative synthetic scenario:** Compare blood pressure before and after treatment (simulated measurements) for the **same people**. Define $d_i=after_i-before_i$.

$H_0:\mu_d=0$; $H_1:\mu_d\ne0$.

$$t_{obs}=\frac{\bar d}{s_d/\sqrt n},\quad df=n-1.$$

**p-value before figure:** $\text{p-value}=P(|T|\ge|t_{obs}|)=2[1-\mathrm{CDF}(|t_{obs}|)]=2\mathrm{SF}(|t_{obs}|)$.

Critical strategy: compare $t_{obs}$ with the two $\alpha/2$ quantiles.

In [ ]:
# Synthetic paired measurements. Keep participant pairing intact.
rng_bp = np.random.default_rng(38)
before = rng_bp.normal(142, 14, size=40)
difference = rng_bp.normal(0, 9, size=40)
difference = difference - difference.mean() - 2.35*difference.std(ddof=1)/np.sqrt(len(difference))
after = before + difference
nd = len(difference)
t_paired = difference.mean() / (difference.std(ddof=1) / np.sqrt(nd))
df_paired = nd - 1
p_value_paired = 2 * st.t.sf(abs(t_paired), df=df_paired)
print(f"Mean change = {difference.mean():.3f}")
print(f"t = {t_paired:.3f}, p-value = {p_value_paired:.3f}")

In [ ]:
show_statistical_test(t_paired, alpha, distribution="t", df=df_paired,
                      alternative="two-sided", title="Paired t-test: mean after - before")
paired_result = st.ttest_rel(after, before)
print(f"SciPy: t = {paired_result.statistic:.3f}, p-value = {paired_result.pvalue:.3f}")
assert np.isclose(paired_result.statistic, t_paired)
assert np.isclose(paired_result.pvalue, p_value_paired)

**Check for understanding (3 min).** Why do we calculate a single array of differences instead of treating the before and after measurements as independent samples?

## 6. One-way ANOVA — 20 minutes

**Illustrative synthetic scenario:** Compare Data Scientist salaries across three company sizes (simulated salaries).

$H_0:\mu_1=\mu_2=\mu_3$. $H_1$: at least one group mean differs (not necessarily all of them).

Let $k$ be the number of groups, $n_j$ the size of group $j$, $N=\sum_{j=1}^{k}n_j$ the total number of observations, $\bar x_j$ the mean of group $j$, and $\bar x$ the grand mean (mean across all observations). An observation $x_{ij}$ is observation $i$ in group $j$.

**Between-group variation:** how far each group mean is from the grand mean, weighted by group size.

$$SS_{between}=\sum_{j=1}^{k}n_j(\bar x_j-\bar x)^2.$$

**Within-group variation:** how far each observation is from its own group mean.

$$SS_{within}=\sum_{j=1}^{k}\sum_{i=1}^{n_j}(x_{ij}-\bar x_j)^2.$$

**Notation in the formulas:**

- $k$: number of groups; $n_j$: number of observations in group $j$; $N$: total number of observations across all groups.
- $x_{ij}$: observation $i$ in group $j$; $\bar{x}_j$: sample mean of group $j$; $\bar{x}$: grand mean calculated from **all observations** (not the unweighted mean of group means when group sizes differ).
- $SS_{between}$: sum of squares **between** groups; $SS_{within}$: sum of squares **within** groups.
- $MS_{between}$ and $MS_{within}$: corresponding mean squares (sums of squares divided by their degrees of freedom).
- $F$: ratio of between-group to within-group mean squares; $k-1$ and $N-k$: their respective degrees of freedom.

Convert sums of squares to mean squares and form the ratio:

$$MS_{between}=\frac{SS_{between}}{k-1},\qquad MS_{within}=\frac{SS_{within}}{N-k},\qquad F=\frac{MS_{between}}{MS_{within}}.$$

The numerator measures differences **between** groups; the denominator measures variability **within** groups.

Here $F\ge0$ and **large values** indicate evidence against $H_0$. The $F$ distribution is **not symmetric**.

**p-value before figure:**

$$\text{p-value}=P(F_{k-1,N-k}\ge F_{obs}\mid H_0)=1-\mathrm{CDF}(F_{obs})=\mathrm{SF}(F_{obs}).$$

**Critical strategy:** reject when $F_{obs}>\mathrm{ISF}(\alpha)$.

Classical ANOVA assumes independent observations, approximately normal errors within groups and equal population variances. ANOVA rejection does **not** tell us which means differ.

In [ ]:
# Synthetic salary groups, with modest between-group differences.
rng_salary = np.random.default_rng(124)
groups = [rng_salary.normal(100_000, 20_000, size=30),
          rng_salary.normal(100_000, 20_000, size=30),
          rng_salary.normal(100_000, 20_000, size=30)]
# Give the group means a modest separation; scale offsets to obtain F ~ 3.7.
base_means = [g.mean() for g in groups]
groups = [g - m for g, m in zip(groups, base_means)]
within = sum(((g-g.mean())**2).sum() for g in groups)
ms_within = within/(90-3)
shift = np.sqrt(3.7*ms_within/30)
groups = [groups[0] - shift, groups[1], groups[2] + shift]
k = len(groups)
N = sum(len(g) for g in groups)
grand_mean = np.concatenate(groups).mean()
ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
ss_within = sum(((g - g.mean())**2).sum() for g in groups)
dfn, dfd = k - 1, N - k
f_obs = (ss_between / dfn) / (ss_within / dfd)
p_value_anova = st.f.sf(f_obs, dfn=dfn, dfd=dfd)
print(f"F = {f_obs:.3f}, df = ({dfn}, {dfd}), p-value = {p_value_anova:.3f}")

In [ ]:
show_statistical_test(f_obs, alpha, distribution="f", df=(dfn, dfd),
                      alternative="greater", title="One-way ANOVA: right-tail F test")
anova_result = st.f_oneway(*groups)
print(f"SciPy: F = {anova_result.statistic:.3f}, p-value = {anova_result.pvalue:.3f}")
assert np.isclose(anova_result.statistic, f_obs)
assert np.isclose(anova_result.pvalue, p_value_anova)

**Check for understanding (3 min).** Why is multiplying `SF(F_obs)` by two incorrect? Does a significant ANOVA prove that all three means differ?

## 7. Chi-square test of independence — optional extension (15 minutes)

*This topic extends the original notebook.* Use a contingency table to test association between two categorical variables (here, workshop attendance and survey response).

$H_0$: the variables are independent. $H_1$: they are associated.

$$\chi^2=\sum_{i,j}\frac{(O_{ij}-E_{ij})^2}{E_{ij}},\qquad E_{ij}=\frac{(row\ total_i)(column\ total_j)}{N}.$$

Degrees of freedom: $(r-1)(c-1)$.

**p-value before figure:** $\text{p-value}=P(\chi^2\ge\chi^2_{obs}\mid H_0)=1-\mathrm{CDF}(\chi^2_{obs})=\mathrm{SF}(\chi^2_{obs})$.

**Critical strategy:** reject when $\chi^2_{obs}>\mathrm{ISF}(\alpha)$. Interpret association, not causation. Inspect expected counts before relying on the approximation.

In [ ]:
# Synthetic 3x2 contingency table: workshop group by response (yes/no).
observed = pd.DataFrame([[38, 62], [52, 48], [56, 44]],
                        index=["Group A", "Group B", "Group C"],
                        columns=["Yes", "No"])
row_totals = observed.sum(axis=1).to_numpy()[:, None]
col_totals = observed.sum(axis=0).to_numpy()[None, :]
expected = row_totals @ col_totals / observed.to_numpy().sum()
chi_obs = (((observed.to_numpy() - expected)**2) / expected).sum()
chi_df = (observed.shape[0] - 1) * (observed.shape[1] - 1)
chi_p_value = st.chi2.sf(chi_obs, df=chi_df)
print(f"Chi-square = {chi_obs:.3f}, df = {chi_df}, p-value = {chi_p_value:.3f}")
print(f"Minimum expected count = {expected.min():.3f}")

In [ ]:
show_statistical_test(chi_obs, alpha, distribution="chi2", df=chi_df,
                      alternative="greater", title="Chi-square test: right-tail area")
chi_result = st.chi2_contingency(observed, correction=False)
print(f"SciPy: chi-square = {chi_result.statistic:.3f}, p-value = {chi_result.pvalue:.3f}")
assert np.isclose(chi_result.statistic, chi_obs)
assert np.isclose(chi_result.pvalue, chi_p_value)

### Cramér's V — how strong is the association?

The chi-square test tells us **whether there is evidence of an association** between two categorical variables. It does not, by itself, quantify **how strong** that association is.

**Cramér's V** is an effect-size measure based on the same chi-square statistic:

$$
V = \sqrt{\frac{\chi^2_{obs}}{N\,\min(r-1,\,c-1)}}
$$

where:

- $\chi^2_{obs}$ is the observed chi-square statistic;
- $N$ is the total number of observations;
- $r$ is the number of rows (categories of the first variable);
- $c$ is the number of columns (categories of the second variable);
- $\min(r-1,c-1)$ is the smaller of the two values.

**Interpretation:** $V$ ranges from **0** (no association in the observed table) to **1** (maximum association). Unlike Pearson's $r$, it has **no positive or negative sign**. The practical meaning of a particular value depends on the context and table dimensions; avoid universal cutoffs for "weak" or "strong" association.

We can compute it directly from the chi-square statistic already obtained, without running another hypothesis test.

In [ ]:
# Effect size for the same contingency table: Cramer's V.
n_total = observed.to_numpy().sum()
n_rows, n_cols = observed.shape
cramers_v = np.sqrt(chi_obs / (n_total * min(n_rows - 1, n_cols - 1)))

print(f"Cramér's V = {cramers_v:.3f}")
print("Chi-square tests independence; Cramér's V describes association strength.")


**Check for understanding (3 min).** Is the chi-square statistic negative? What does a small p-value tell us about independence?

## 8. Comparing two independent proportions — smoking cessation (optional, 20 minutes)

A public-health team follows two **independent** groups of participants in a smoking-cessation programme. In this **illustrative dataset**, **72 of 400 men** and **100 of 400 women** report that they stopped smoking at the agreed follow-up date. Each participant has a binary outcome: stopped smoking (1) or did not (0).

A *cessation rate* is a **proportion**, not a mean of a continuous variable. Let $p_m$ and $p_w$ be the population probabilities of cessation for men and women. We want to test whether these probabilities **differ**, without assuming a direction.

Start with the hypotheses in terms of population proportions:

$$H_0:p_w=p_m,\qquad H_1:p_w\ne p_m.$$

Rewrite them using a hypothesized difference $\delta$:

$$H_0:p_w-p_m=\delta,\qquad H_1:p_w-p_m\ne\delta.$$

**For this example, $\delta=0$**: the null hypothesis states that the cessation probabilities are equal. A two-sided alternative is appropriate because either group could have the higher cessation rate.

The sample proportions are $\hat p_m=x_m/n_m$ and $\hat p_w=x_w/n_w$. **Only under the equality null ($\delta=0$)**, pool the successes to estimate the common population probability:

$$\hat p_{pool}=\frac{x_m+x_w}{n_m+n_w}.$$

The **two-proportion z statistic for the equality null** is

$$z_{obs}=\frac{(\hat p_w-\hat p_m)-\delta}{\sqrt{\hat p_{pool}(1-\hat p_{pool})(1/n_m+1/n_w)}},\qquad \delta=0.$$

Here $x_m,x_w$ are the numbers of successes, and $n_m,n_w$ are the group sizes. **Important:** writing $\delta$ makes the null difference explicit, but this *pooled* standard error is valid for $\delta=0$. Testing a nonzero hypothesized difference requires a different variance calculation; do not simply change $\delta$ in the code below.

With independent observations and sufficiently large expected counts, $Z$ is approximately standard normal. For the **two-sided** alternative:

$$\text{p-value}=P(|Z|\ge|z_{obs}|\mid H_0)=2\,P(Z\ge|z_{obs}|)=2[1-\mathrm{CDF}(|z_{obs}|)]=2\,\mathrm{SF}(|z_{obs}|).$$

**Critical-value strategy:** reject $H_0$ if $|z_{obs}|>\mathrm{ISF}(\alpha/2)$. **p-value strategy:** reject when p-value $<\alpha$. The Matplotlib figure highlights both rejection tails.

**Interpretation:** an association between gender and cessation in observational data does **not** establish that gender causes a difference. Other factors may affect cessation rates.

In [ ]:
# Illustrative smoking-cessation outcomes (independent groups)
n_m, x_m = 400, 72
n_w, x_w = 400, 100
delta = 0.0  # H0: p_w - p_m = delta; pooled test requires delta = 0
rate_m = x_m / n_m
rate_w = x_w / n_w
pooled_rate = (x_m + x_w) / (n_m + n_w)
standard_error = np.sqrt(pooled_rate * (1 - pooled_rate) * (1/n_m + 1/n_w))
z_obs_proportions = ((rate_w - rate_m) - delta) / standard_error
p_value_proportions = 2 * st.norm.sf(abs(z_obs_proportions))
z_critical = st.norm.isf(alpha / 2)
print(f"Men: cessation rate = {rate_m:.3f}")
print(f"Women: cessation rate = {rate_w:.3f}")
print(f"Observed difference (women - men) = {rate_w - rate_m:.3f}")
print(f"z = {z_obs_proportions:.3f}, critical |z| = {z_critical:.3f}, p-value = {p_value_proportions:.3f}")
print(f"Decision by critical value: {'Reject H0' if abs(z_obs_proportions) > z_critical else 'Do not reject H0'}")
print(f"Decision by p-value: {'Reject H0' if p_value_proportions < alpha else 'Do not reject H0'}")
show_statistical_test(z_obs_proportions, alpha, distribution="normal", alternative="two-sided",
                      title="Smoking cessation: two-sided test of independent proportions")
# SciPy cross-check: Pearson chi-square on a 2x2 table (without Yates correction)
counts = np.array([[x_m, n_m-x_m], [x_w, n_w-x_w]])
chi2_result = st.chi2_contingency(counts, correction=False)
assert np.isclose(chi2_result.statistic, z_obs_proportions**2)
assert np.isclose(chi2_result.pvalue, p_value_proportions)


**Check for understanding (4 min).** Why is this a two-sided test? Why is the pooled rate appropriate when $\delta=0$? Would a statistically significant result demonstrate that gender *causes* a difference in smoking-cessation success?

## 9. Final recap — 10–15 minutes

A product team uses **30 seconds** as its benchmark for average checkout time. You want to investigate whether the true average checkout time **differs** from that benchmark. A random sample of 36 independent checkouts has a mean of 32 seconds and a sample standard deviation of 6 seconds. Assume the t-test is appropriate. Use $\alpha=0.05$.

1. State $H_0:\mu=30$ and the **two-sided** alternative $H_1:\mu\ne30$. Explain why the equality belongs in $H_0$ rather than $H_1$.
2. Calculate the test statistic **by hand** and identify the degrees of freedom.
3. Express the p-value as a probability under $H_0$, then using `CDF` and `SF`.
4. Find both critical values with `ppf` and/or `isf`. Decide using critical values.
5. Decide using the p-value. Confirm the two decisions agree.
6. Use `show_statistical_test()` to display the two rejection regions and observed tails.
7. Write a one-sentence conclusion **without** saying the null hypothesis is true or false.

**Extension:** Repeat with $H_1:\mu>30$. How do the p-value and critical region change?

In [ ]:
# Final recap — write your solution here.
# Use scipy.stats.t for the theoretical distribution.

### Before leaving

- A **test statistic** locates our sample on a null distribution.
- A **critical value** is the boundary defined by alpha.
- A **p-value** is the null-tail probability of a result at least as extreme as observed.
- Both decision methods are equivalent when they describe the **same** test.
- A non-significant result is **not proof of equality**.

**Practice:** Complete the original Hypothesis Testing lab after class.